# Week 1 — URL Summarizer

Calls the OpenAI API to summarize the contents of a given website URL.

**Stack:** Python, OpenAI API, BeautifulSoup (via `scraper`), dotenv

In [ ]:
# Imports
import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

## Load environment variables and validate API key

In [ ]:
# Load .env and pull the OpenAI API key
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Basic sanity checks on the key
if not api_key:
    print("No API key found — check your .env file.")
elif not api_key.startswith("sk-proj-"):
    print("API key found, but doesn't start with 'sk-proj-' — verify it's the correct key.")
elif api_key.strip() != api_key:
    print("API key has leading/trailing whitespace — strip it.")
else:
    print("API key looks good.")

## Quick test call to OpenAI

In [ ]:
# Minimal example call to verify the client works
openai = OpenAI()

messages = [{"role": "user", "content": "Hello, GPT! This is a test message."}]
response = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
print(response.choices[0].message.content)

## Define system and user prompts for summarization

In [ ]:
# System prompt — sets the assistant's role and tone
system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

# User prompt prefix — appended with the scraped website content
user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

## Helper functions

In [ ]:
# Build the OpenAI messages list from scraped website content
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [ ]:
# Fetch a URL, send to OpenAI, return the model's summary
def summarize(url):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages_for(website)
    )
    return response.choices[0].message.content

In [ ]:
# Render the summary as markdown in the notebook
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

## Run the summarizer

In [ ]:
# Test on a few public sites
display_summary("https://cnn.com")

In [ ]:
display_summary("https://anthropic.com")

## Notes

- Only works on websites that can be scraped via simple HTTP fetch + HTML parsing.
- JavaScript-rendered sites (React apps, etc.) won't work — would need Selenium or Playwright.
- CloudFront-protected sites may return 403.